This notebook checks the code for importing the disk dictionary and serves as a place to document notes and sources for the values included in the dictionary.

In [1]:
# add the host disk properties 
import pickle
import os
import numpy as np
with open(r'd:\CPD_MPIA\CPD_Emission_Models\disk_arr.pkl', 'rb') as f:
    disk_arr = pickle.load(f)

In [2]:
# Add the rms (mu Jy/beam) (-0.5)
disk_rms = {
    "DM Tau": 26.6,
    "AA Tau": 23.8,
    "LkCa 15": 20.3,
    "HD 34282": 22.6,
    "MWC 758": 31.6,
    "CQ Tau": 25.5,
    "SY Cha": 30.7,
    "PDS 66": 26.0,
    "HD 135344B": 23.4,
    "HD 143006": 25.3,
    "J1604": 23.0,
    "J1615": 19.1,
    "V4046 Sgr": 19.7,
    "J1842": 23.4,
    "J1852": 19.7
}

In [3]:
# Calculate rout as defined by the radius where brightness drops to 2*rms, and rms is calculated as the median of the MAD between 3*R90 and 5*R90
robust_value = 2.0  

def calculate_rms_median_mad(profile, r90):
    # profile: array of ( radius [arcsec] intensity [Jy/beam] standard deviation [Jy/beam] )
    # r90: float, R90 value
    # Select radii between 3*R90 and 5*R90
    mask = (profile[:,0] >= 3*r90) & (profile[:,0] <= 5*r90)
    selected_dy = profile[mask, 2]
    RMS = np.median(selected_dy)*1e6   # Convert to muJy/beam
    return RMS

def find_rout(profile, rms):
    # Find radius where brightness drops to 2*rms
    for radius, brightness, stdev, clean_brightness in profile:
        if clean_brightness <= 2*rms/1e6:  # Convert rms back to Jy/beam
            return radius
    return None  # If not found


In [4]:
import sys
sys.path.append(r'D:\CPD_MPIA\Median_SNR')
import pickle

with open(r'd:\CPD_MPIA\Median_SNR\all_disks.pkl', "rb") as f:
    all_disks = pickle.load(f)

In [5]:
# Set gthres as 1*rms average at the gap

def calculate_rms_at_gaps(profile, rgap_list, wgap_list):
    """
    Calculate RMS at gap regions
    
    Parameters:
    - profile: array of (radius [arcsec], intensity [Jy/beam], standard deviation [Jy/beam])
    - rgap_list: list of gap center radii in arcsec
    - wgap_list: list of gap widths in arcsec
    
    Returns:
    - RMS in muJy/beam calculated from standard deviations in gap regions
    """
    gap_mask = np.zeros(len(profile), dtype=bool)
    
    # Create mask for all gap regions
    for rgap, wgap in zip(rgap_list, wgap_list):
        # Define gap boundaries with buffer
        gap_inner = rgap - (wgap/2)
        gap_outer = rgap + (wgap/2) 
        
        # Add this gap region to the mask
        gap_region_mask = (profile[:,0] >= gap_inner) & (profile[:,0] <= gap_outer)
        gap_mask |= gap_region_mask
    
    if not np.any(gap_mask):
        # No data in gap regions, fallback to median of all data
        print("Warning: No data found in gap regions, using all data")
        selected_dy_gap = profile[:, 2]
    else:
        # Use standard deviations from gap regions
        selected_dy_gap = profile[gap_mask, 2]
    
    RMS_gap = np.median(selected_dy_gap) * 1e6  # Convert to muJy/beam
    return RMS_gap

In [6]:
def calculate_rms_individual_gaps(profile, rgap_list, wgap_list):
    """
    Calculate RMS for each gap individually
    
    Parameters:
    - profile: array of (radius [arcsec], intensity [Jy/beam], standard deviation [Jy/beam])
    - rgap_list: list of gap center radii in arcsec
    - wgap_list: list of gap widths in arcsec
    
    Returns:
    - List of RMS values in muJy/beam, one for each gap
    """
    gap_rms_list = []
    
    for rgap, wgap in zip(rgap_list, wgap_list):
        # Define gap boundaries
        gap_inner = rgap - (wgap/2)
        gap_outer = rgap + (wgap/2)
        
        # Create mask for this specific gap
        gap_mask = (profile[:,0] >= gap_inner) & (profile[:,0] <= gap_outer)
        
        if not np.any(gap_mask):
            # No data in this gap, use overall RMS as fallback
            selected_dy = profile[:, 2]
        else:
            # Use standard deviations from this gap region
            selected_dy = profile[gap_mask, 2]
        
        gap_rms = np.median(selected_dy) * 1e6  # Convert to muJy/beam
        gap_rms_list.append(gap_rms)
    
    return gap_rms_list

In [7]:
def build_disk_dicts_for_robust(robust_value):
    all_disk_dicts = {}
    for disk_name, disk_obj in all_disks.items():
        d = {}
        d['name'] = disk_name
        d['label'] = disk_name.replace('_', ' ')
        d['distance'] = getattr(disk_obj, 'distance_pc', None)
        d['incl'] = getattr(disk_obj, 'inc', None)
        d['PA'] = getattr(disk_obj, 'PA', None)
        d['dx'], d['dy'] = getattr(disk_obj, 'center', (None, None))

        # load the radial profile
        profile_path = fr'D:\CPD_MPIA\Median_SNR\Disk_Residual_Profile_Median_SNR\{disk_name}\{disk_name}_residual_radial_profile_robust{robust_value}.txt'
        if not os.path.exists(profile_path):
            print(f"Skipping {disk_name} for robust {robust_value}: file not found.")
            continue
        profile = np.loadtxt(profile_path)
        r90 = disk_obj.disksize["R90"] # float
        rms = calculate_rms_median_mad(profile, r90)
        rout = find_rout(profile, 2*rms)
        d['R90'] = r90
        d['RMS'] = rms
        d['rout'] = rout

        # Add info from disk_arr if available
        if disk_name in disk_arr:
            arr = disk_arr[disk_name]
            d['lstar'] = arr[3]
            d['mstar'] = arr[1]
            d['cthresh'] = f"{5 * d['RMS']/1000:.3f}mJy"  # Convert to string with units

        if hasattr(disk_obj, 'ringgap_info') and "flag" in disk_obj.ringgap_info:
            gaps = disk_obj.ringgap_info['flag'] == 0
            d['rgap'] = (disk_obj.ringgap_info['radius_arcsec'][gaps]).tolist()
            d['wgap'] = (disk_obj.ringgap_info['width_arcsec'][gaps]).tolist()
            d['dgap'] = (disk_obj.ringgap_info['gap_depth'][gaps]*100).tolist()
            
            # Calculate individual gap thresholds - one for each gap!
            if d['rgap'] and d['wgap']:
                d['gscales'] = [0, 5]
                gap_rms_list = calculate_rms_individual_gaps(profile, d['rgap'], d['wgap'])
                #d['gthresh'] = [f"{rms/1000:.3f}mJy" for rms in gap_rms_list]  # Convert to strings with units
                d['gthresh'] = f"{2 * d['RMS']/1000:.3f}mJy"
            else:
                d['gthresh'] = []
        else:
            d['rgap'] = []
            d['wgap'] = []
            d['dgap'] = []
            d['gthresh'] = []
            
        # Add Frank fitting parameters and CASA properties
        d['hyp-alpha'] = 1.3
        d['hyp-wsmth'] = 0.1
        d['hyp-Ncoll'] = 300
        d['cscale'] = [0, 8, 15, 30, 80]
        d['crobust'] = robust_value
        d['ctaper'] = []
        d['cgain'] = 0.3
        d['ccycleniter'] = 300
        # seems like 3 arcsec is too big, so let it be 2 arcsec
        d['cmask'] = (
            f"ellipse[[{arr[6].replace(' ',':')}, {arr[7].replace(' ',':')}], "
            f"[2arcsec, {float(2*np.cos(np.radians(d['incl'])))}arcsec], "
            f"{float(d['PA'])}deg]"
        )


        all_disk_dicts[disk_name] = d

    return all_disk_dicts

robust_values = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
disk_dicts_by_robust = {}

for robust_value in robust_values:
    robust_str = f"m{abs(robust_value)}" if robust_value < 0 else f"{robust_value}"
    robust_str = robust_str.replace('.', '_')
    disk_dicts_by_robust[robust_str] = build_disk_dicts_for_robust(robust_value)
    filename = f"all_disk_dicts_r{robust_str}.pkl"
    with open(filename, "wb") as f:
        pickle.dump(disk_dicts_by_robust[robust_str], f)

Skipping AA_Tau for robust 1.5: file not found.


In [8]:


# Now you can access the dictionary:
#print(all_disk_dicts["AA_Tau"])

#### check with the Andrews Code   (CASA paramters use exoALMAII)

```
disk = {}

disk['SR4'] =      {'name': 'SR4',   
                    'label': 'SR 4', 
                    'distance': 134.8,
                    'mstar': 0.68,
                    'lstar': 1.17,
                    'incl': 22.0, 
                    'PA': 18.0,
                    'dx': -0.060,  # RA offset in arcsec
                    'dy': -0.509,  # Dec offset in arcsec
                    'rgap': [0.079],  # mas
                    'wgap': [0.010],  # mas
                    'dgap': [30],  # 30 percent flux drop
                    'rout': 0.25, 
                    'maxTb': 50,
                    'hyp-alpha': 1.3,   # frank fitting  - smoothing strength
                    'hyp-wsmth': 0.1, # frank fitting  - smoothing scale
                    'hyp-Ncoll': 300,  # frank fitting  - number of collocation points
                    'cmask': 'circle[[16h25m56.16s, -24.20.48.71], 0.7arcsec]',  # for casa tclean to define imaging region (not full fov)
                    'cscales': [0, 5, 30, 75, 150],    # in very simple words, these are the different "size" of structures that tclean will try to decompose the image into, structures of size ~scale will be modelled with that scale, in units of pixels
		    'gscales': [0, 5],   # for gap specific cleaning
                    'cthresh': '0.05mJy',   
                    'gthresh': '0.034mJy',
                    'crobust': -0.5,
                    'ctaper': ['0.035arcsec', '0.01arcsec', '0deg'],
                    'cgain': 0.3,
                    'ccycleniter': 300,
                    'RMS': 17.8,
                    'peakr': [84.], 
                    'peakaz': [104.]
}
```

In [9]:
# To print and check individually:
import pprint
for key in disk_dicts_by_robust:
    print(f"Robust value: {key}")
    pprint.pprint(disk_dicts_by_robust[key])
    print("\n")

Robust value: m2_0
{'AA_Tau': {'PA': 93.77079777,
            'R90': np.float64(1.035),
            'RMS': np.float64(49.17835394735448),
            'ccycleniter': 300,
            'cgain': 0.3,
            'cmask': 'ellipse[[04:34:55.420, +24:28:53.034], [2arcsec, '
                     '1.0439459414332437arcsec], 93.77079777deg]',
            'crobust': -2.0,
            'cscale': [0, 8, 15, 30, 80],
            'ctaper': [],
            'cthresh': '0.246mJy',
            'dgap': [1.0, 44.0, 34.0, 94.0],
            'distance': 135,
            'dx': -0.00545897,
            'dy': 0.00482739,
            'gscales': [0, 5],
            'gthresh': '0.098mJy',
            'hyp-Ncoll': 300,
            'hyp-alpha': 1.3,
            'hyp-wsmth': 0.1,
            'incl': 58.53531224,
            'label': 'AA Tau',
            'lstar': 1.1,
            'mstar': 0.79,
            'name': 'AA_Tau',
            'rgap': [0.082, 0.478, 0.593, 0.782],
            'rout': np.float64(0.09441259969

In [10]:
import pprint

for key in disk_dicts_by_robust:
    if key != "0_5":
        continue
    print(f"Robust value: {key}")
    pprint.pprint(disk_dicts_by_robust[key])
    print("\n")

Robust value: 0_5
{'AA_Tau': {'PA': 93.77079777,
            'R90': np.float64(1.035),
            'RMS': np.float64(26.37405850691721),
            'ccycleniter': 300,
            'cgain': 0.3,
            'cmask': 'ellipse[[04:34:55.420, +24:28:53.034], [2arcsec, '
                     '1.0439459414332437arcsec], 93.77079777deg]',
            'crobust': 0.5,
            'cscale': [0, 8, 15, 30, 80],
            'ctaper': [],
            'cthresh': '0.132mJy',
            'dgap': [1.0, 44.0, 34.0, 94.0],
            'distance': 135,
            'dx': -0.00545897,
            'dy': 0.00482739,
            'gscales': [0, 5],
            'gthresh': '0.053mJy',
            'hyp-Ncoll': 300,
            'hyp-alpha': 1.3,
            'hyp-wsmth': 0.1,
            'incl': 58.53531224,
            'label': 'AA Tau',
            'lstar': 1.1,
            'mstar': 0.79,
            'name': 'AA_Tau',
            'rgap': [0.082, 0.478, 0.593, 0.782],
            'rout': np.float64(1.1712189791721

In [15]:
for key in disk_dicts_by_robust:
    if key != "1_5":
        continue
    print(f"Robust value: {key}")
    pprint.pprint(disk_dicts_by_robust[key])
    print("\n")

Robust value: 1_5
{'CQ_Tau': {'PA': 53.87180444,
            'R90': np.float64(0.4894),
            'RMS': np.float64(25.781127988011576),
            'ccycleniter': 300,
            'cgain': 0.3,
            'cmask': 'ellipse[[05:35:58.467, +24:44:54.091], [2arcsec, '
                     '1.633432101799876arcsec], 53.87180444deg]',
            'crobust': 1.5,
            'cscale': [0, 8, 15, 30, 80],
            'ctaper': [],
            'cthresh': '0.129mJy',
            'dgap': [],
            'distance': 149,
            'dx': -0.00871044,
            'dy': 0.0009941,
            'gthresh': [],
            'hyp-Ncoll': 300,
            'hyp-alpha': 1.3,
            'hyp-wsmth': 0.1,
            'incl': 35.2426038,
            'label': 'CQ Tau',
            'lstar': 10,
            'mstar': 1.4,
            'name': 'CQ_Tau',
            'rgap': [],
            'rout': np.float64(0.868265256285675),
            'wgap': []},
 'DM_Tau': {'PA': 155.59975598,
            'R90': np.float

In [12]:
# kinks, what is different